#### 문항 1. 네이버 연관검색어 수집 함수 만들기

키워드를 입력받아 그 키워드의 연관검색어 리스트를 돌려주는 함수를 작성하시오.

* 함수명: get_related_keywords(keyword)

* 반환: 문자열 리스트

* Selenium 사용 금지. Network 탭에서 요청을 찾아 requests로 재현할 것

* 결과가 없으면 빈 리스트를 돌려줄 것 (예외를 던지지 말 것)

* 힌트.md에 문서 찾기 힌트, 결과 파싱 힌트 있음

실행 예시
```
>>>get_related_keywords('부트캠프')
['부트캠프 뜻', '부트캠프', '부트캠프 추천', '카카오 부트캠프', '맥북 부트캠프',
 '카카오테크 부트캠프', '직무부트캠프', '네이버 부트캠프', '마케팅 부트캠프', '코멘토 직무부트캠프']
 ```

In [2]:
import re
import json
import requests


def get_related_keywords(keyword):

    url = "https://ac.search.naver.com/nx/ac"

    params = {
        "q": keyword,  
        "st": 100,      
        "frm": "nv",   
        "q_enc": "UTF-8",
        "r_enc": "UTF-8",
    }

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        "Referer": "https://search.naver.com/",
    }

    try:
        res = requests.get(url, params=params, headers=headers, timeout=5)
        res.raise_for_status()  

        raw_text = res.text

        match = re.search(r"\{.*\}", raw_text, re.S)
        if not match:
            return []

        data = json.loads(match.group())

        items = data.get("items")
        if not items or not items[0]:
            return []

        keywords = [entry[0] for entry in items[0] if entry]
        return keywords

    except Exception:

        return []


if __name__ == "__main__":
    result = get_related_keywords("부트캠프")
    print(result)

['부트캠프', '부트캠프 뜻', '부트캠프 취업', 'ai 부트캠프', '직무부트캠프', '코햄 부트캠프', '마케팅 부트캠프', '맥북 부트캠프', '코멘토 부트캠프', '넷플릭스 부트캠프']


#### 문항 2. 네이버 웹툰 전체 목록 수집

네이버 웹툰의 요일별 전체 웹툰을 수집하시오.

* 대상: https://comic.naver.com/webtoon

* 추출 필드: 제목 / 링크 / 요일

* 모든 요일의 웹툰을 수집할 것

* 링크는 상세 페이지로 바로 이동할 수 있는 절대 주소로 만들 것 (힌트 1)

* 결과를 naver_webtoon.csv로 저장할 것

결과 예시
```
[{'제목': '광마회귀', '링크': 'https://comic.naver.com/webtoon/list?titleId=776601', '요일': '금'},
 {'제목': '외모지상주의', '링크': 'https://comic.naver.com/webtoon/list?titleId=641253', '요일': '금'},
 ...]
```

In [ ]:
import csv
import requests

API_URL = "https://comic.naver.com/api/webtoon/titlelist/weekday"
DETAIL_URL = "https://comic.naver.com/webtoon/list?titleId={title_id}"

WEEKDAY_KOR = {
    "MONDAY": "월",
    "TUESDAY": "화",
    "WEDNESDAY": "수",
    "THURSDAY": "목",
    "FRIDAY": "금",
    "SATURDAY": "토",
    "SUNDAY": "일",
}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Referer": "https://comic.naver.com/webtoon",
}


def get_all_webtoons():
    res = requests.get(API_URL, params={"order": "user"}, headers=HEADERS, timeout=10)
    res.raise_for_status()
    data = res.json()

    title_list_map = data.get("titleListMap", {})

    results = []
    for eng_weekday, webtoon_list in title_list_map.items():
        kor_weekday = WEEKDAY_KOR.get(eng_weekday, eng_weekday)

        for webtoon in webtoon_list:
            title_id = webtoon.get("titleId")
            title_name = webtoon.get("titleName")

            if not title_id or not title_name:
                continue

            results.append({
                "제목": title_name,
                "링크": DETAIL_URL.format(title_id=title_id),
                "요일": kor_weekday,
            })

    return results


def save_to_csv(data, filename="naver_webtoon.csv"):
    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=["제목", "링크", "요일"])
        writer.writeheader()
        writer.writerows(data)


if __name__ == "__main__":
    webtoons = get_all_webtoons()
    save_to_csv(webtoons)
    print(f"{len(webtoons)}개 웹툰 저장 완료 -> naver_webtoon.csv")
    print(webtoons[:5])

778개 웹툰 저장 완료 -> naver_webtoon.csv
[{'제목': '나노마신', '링크': 'https://comic.naver.com/webtoon/list?titleId=747271', '요일': '목'}, {'제목': '마흔 즈음에', '링크': 'https://comic.naver.com/webtoon/list?titleId=836052', '요일': '목'}, {'제목': '육아일기', '링크': 'https://comic.naver.com/webtoon/list?titleId=812354', '요일': '목'}, {'제목': '재벌집 막내아들', '링크': 'https://comic.naver.com/webtoon/list?titleId=800770', '요일': '목'}, {'제목': '소꿉친구 컴플렉스', '링크': 'https://comic.naver.com/webtoon/list?titleId=822640', '요일': '목'}]


#### 문항 3. 사람인 채용공고 10페이지 수집

사람인의 공개 채용공고 목록을 10페이지 수집하시오.

* 대상: https://www.saramin.co.kr/zf_user/jobs/public/list

* 추출 필드: 기업명 / 그룹사 / 기업종류 / 공고명 / 직무키워드 / 학력 / 경력구분 / 근무지

* 직무키워드는 리스트로 담을 것

* 값이 없는 항목은 빈 문자열 또는 빈 리스트로 둘 것 (오류로 멈추지 말 것)

* 요청 간 0.5초 이상 지연을 둘 것

* 결과를 saramin.csv로 저장할 것

결과 예시
```
[{'기업명': '(주)유니드',
  '그룹사': '오씨아이그룹',
  '기업종류': '대기업',
  '공고명': '2025년 유니드 상반기 신입사원 수시채용',
  '직무키워드': ['외환관리', '자금관리', '자산운용', '재무제표', '재무회계'],
  '학력': '대학교(4년)↑',
  '경력구분': '신입 · 정규직',
  '근무지': '서울 중구 외'},
 ...]
 ```

In [ ]:
import csv
import time
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.saramin.co.kr"
LIST_URL = f"{BASE_URL}/zf_user/jobs/public/list"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
}


def _text_or_default(tag, default=""):
    if tag is None:
        return default
    text = tag.get_text(strip=True)
    return text if text else default


def parse_one_item(item):
    result = {
        "기업명": "",
        "그룹사": "",
        "기업종류": "",
        "공고명": "",
        "직무키워드": [],
        "학력": "",
        "경력구분": "",
        "근무지": "",
    }

    try:
        company_area = item.select_one(".col.company_nm")
        if company_area:
            name_tag = company_area.select_one("a.str_tit")
            result["기업명"] = _text_or_default(name_tag)

            group_tag = company_area.select_one(".main_corp")
            result["그룹사"] = _text_or_default(group_tag)

            type_tag = company_area.select_one(".info_stock")
            result["기업종류"] = _text_or_default(type_tag)
    except Exception:
        pass

    try:
        info_area = item.select_one(".col.notification_info")
        if info_area:
            title_tag = info_area.select_one(".job_tit a.str_tit")
            result["공고명"] = _text_or_default(title_tag)

            sector_area = info_area.select_one(".job_meta .job_sector")
            if sector_area:
                keywords = [
                    s.get_text(strip=True) for s in sector_area.find_all("span")
                ]
                result["직무키워드"] = [k for k in keywords if k]
    except Exception:
        pass

    try:
        condition_area = item.select_one(".col.recruit_info")
        if condition_area:
            result["근무지"] = _text_or_default(
                condition_area.select_one("p.work_place")
            )
            result["경력구분"] = _text_or_default(
                condition_area.select_one("p.career")
            )
            result["학력"] = _text_or_default(
                condition_area.select_one("p.education")
            )
    except Exception:
        pass

    return result

def get_saramin_jobs(pages=10):
    all_jobs = []

    for page in range(1, pages + 1):
        try:
            res = requests.get(
                LIST_URL,
                params={"page": page},
                headers=HEADERS,
                timeout=10,
            )
            res.raise_for_status()
        except Exception as e:
            print(f"[경고] {page} 페이지 요청 실패: {e}")
            time.sleep(0.5)
            continue

        soup = BeautifulSoup(res.text, "html.parser")
        items = soup.select(".list_item")

        for item in items:
            all_jobs.append(parse_one_item(item))

        print(f"{page} 페이지 완료 (누적 {len(all_jobs)}건)")

        time.sleep(0.5) 

    return all_jobs

def save_to_csv(data, filename="saramin.csv"):
    fieldnames = [
        "기업명", "그룹사", "기업종류", "공고명",
        "직무키워드", "학력", "경력구분", "근무지",
    ]
    with open(filename, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in data:
            row = dict(row)
            row["직무키워드"] = ", ".join(row["직무키워드"])  
            writer.writerow(row)

if __name__ == "__main__":
    jobs = get_saramin_jobs(pages=10)
    save_to_csv(jobs)
    print(f"\n총 {len(jobs)}건 저장 완료 -> saramin.csv")
    print(jobs[:3])

1 페이지 완료 (누적 20건)
2 페이지 완료 (누적 40건)
3 페이지 완료 (누적 60건)
4 페이지 완료 (누적 80건)
5 페이지 완료 (누적 100건)
6 페이지 완료 (누적 120건)
7 페이지 완료 (누적 140건)
8 페이지 완료 (누적 160건)
9 페이지 완료 (누적 180건)
10 페이지 완료 (누적 200건)

총 200건 저장 완료 -> saramin.csv
[{'기업명': '양산부산대학교병원', '그룹사': '', '기업종류': '', '공고명': '2026년도 8월 블라인드 공개채용 계약직(업무(시설)지원직) 모집', '직무키워드': ['시설관리', '대학병원', '일반병원', '종합병원', '기전기사'], '학력': '학력무관', '경력구분': '신입 · 계약직', '근무지': '경남 양산시'}, {'기업명': '양산부산대학교병원', '그룹사': '', '기업종류': '', '공고명': '2026년도 8월 블라인드 공개채용 계약직(간호조무직) 모집', '직무키워드': ['간호조무사', '외래', '대학병원', '일반병원', '종합병원'], '학력': '학력무관', '경력구분': '신입 · 계약직', '근무지': '경남 양산시'}, {'기업명': '양산부산대학교병원', '그룹사': '', '기업종류': '', '공고명': '2026년도 8월 블라인드 공개채용 계약직(행정직) 모집', '직무키워드': ['사무직', '서무', '문서작성', '사무보조', '사무행정'], '학력': '학력무관', '경력구분': '신입 · 계약직', '근무지': '경남 양산시'}]
